# SCM-VLM on Kaggle (free GPU)

Before you start, set these in the right-hand panel under **Session options**:

* **Accelerator → GPU T4 x2** (or GPU P100)
* **Internet → On**. This needs a phone-verified Kaggle account.

No Kaggle dataset is needed: cell 3 downloads the data straight from the official VQA and COCO
sites.

A session lasts at most 12 hours, and only `/kaggle/working` is kept, when you click
**Save Version**. Download the results zip (last cell) before the session ends.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("CUDA available:", torch.cuda.is_available())

## 2. Get the code

In [ ]:
%cd /kaggle/working
!rm -rf scm
!git clone https://github.com/MANDEEP-SAI-SURYA/SCM-VLM-Discrete-Graph.git scm
%cd /kaggle/working/scm
!git log --oneline | head -3
!pip install -q transformers timm networkx seaborn

## 3. Download the data (about 10 minutes, 1.6 GB)

This downloads the VQA v2 questions and answers, the COCO instance annotations, and a **fixed
subset of 8000 COCO images** (seed 0, so every run uses the same images). The subset has
about 43,000 questions. If anything fails, just run the cell again: files that are already
downloaded are skipped.

In [ ]:
%cd /kaggle/working/scm
!python download_data.py --images 8000
!python validate_dataset.py

## 4. Tests (CPU, about 10 s): should print `103 passed`

In [ ]:
!python -m pytest -q tests

## 5. Training runs

Each run first fills the DETR/BERT feature cache (the first run is the slowest). Then it trains,
evaluates on the held-out split, and writes `results/eval_summary_<tag>.json`.

If time is short, run the cells in this order and stop when needed: `discrete`, `soft`,
`fw_all`, `adaptive`, `neurosym`. You can also use `--samples 15000 --epochs 6`.

In [ ]:
!python train.py --samples 30000 --epochs 10 --graph-mode discrete --tag discrete

In [ ]:
!python train.py --samples 30000 --epochs 10 --graph-mode soft --tag soft

In [ ]:
!python train.py --samples 30000 --epochs 10 --graph-mode discrete --loss-weighting uncertainty --curriculum full --knowledge-prior builtin --symbolic-fusion --tag fw_all

In [ ]:
!python train.py --samples 30000 --epochs 10 --graph-mode discrete --loss-weighting uncertainty --curriculum full --tag adaptive

In [ ]:
!python train.py --samples 30000 --epochs 10 --graph-mode discrete --knowledge-prior builtin --symbolic-fusion --tag neurosym

## 6. Compare the runs

In [ ]:
import json, os, pandas as pd

def row(tag):
    s = json.load(open(f'results/eval_summary_{tag}.json'))
    cf = s.get('counterfactual_accuracy', {})
    r = {'run': tag, 'factual acc': s['factual_accuracy']['overall'],
         'relational acc': s['factual_accuracy'].get('relational'),
         'counterfactual acc': cf.get('overall'),
         **{f'cf {t}': cf.get(t) for t in ['existence', 'counting', 'replacement', 'irrelevant']}}
    g = s.get('graph_structure', {})
    for k in ['dag', 'acyclicity_h', 'root_fraction', 'parents_per_node', 'stability']:
        if k in g: r[k] = g[k]['mean']
    return r

tags = [t for t in ['soft', 'discrete', 'adaptive', 'neurosym', 'fw_all'] if os.path.isfile(f'results/eval_summary_{t}.json')]
display(pd.DataFrame([row(t) for t in tags]).set_index('run').T.round(2))

## 7. Save the results (download the zip from the file browser on the right, under /kaggle/working)

In [ ]:
%cd /kaggle/working
!zip -q -r scm_results.zip scm/results scm/checkpoints
!ls -lh scm_results.zip